In [ ]:
%run -i ../../python_scripts/nb_setup.py

### Diabetes 30-day hospital readmission (tabular)

**Task.** UCI *Diabetes 130-US hospitals (1999-2008)*, 101,766 patients, 50 features. Label: readmitted in under 30 days (~11% positives).
Model: `HistGradientBoostingClassifier`.

**Requirements.** `diabetic_data.csv` from the UCI zip
(<https://archive.ics.uci.edu/dataset/296>), or `pip install ucimlrepo` to fetch
it in-notebook.

In [7]:
LOCAL_CSV = "diabetic_data.csv"  # if absent, fetched via ucimlrepo
TOP_LEVELS = 50  # cap on categorical levels (diag_* has 700+ ICD codes)
SEED, OUT = 0, "sgp_set_tabular_SR"
rng = np.random.default_rng(SEED)

In [ ]:
if os.path.exists(LOCAL_CSV):
    df = pd.read_csv(LOCAL_CSV, na_values="?")
else:
    from ucimlrepo import fetch_ucirepo

    r = fetch_ucirepo(id=296)
    print(r.metadata.name)  # check: "Diabetes 130-US Hospitals for Years 1999-2008"
    parts = [getattr(r.data, "ids", None), r.data.features, r.data.targets]
    df = pd.concat([x for x in parts if x is not None], axis=1)

if "patient_nbr" in df:
    df = df.drop_duplicates("patient_nbr")
if "discharge_disposition_id" in df:  # expired / hospice
    df = df[~df.discharge_disposition_id.isin([11, 13, 14, 19, 20, 21])]

y = (df.readmitted.astype(str).str.strip() == "<30").to_numpy().astype(int)
X = df.drop(
    columns=[
        c for c in ["encounter_id", "patient_nbr", "weight", "readmitted"] if c in df
    ]
)
for c in X.select_dtypes("object"):
    top = X[c].value_counts().index[:TOP_LEVELS]
    X[c] = X[c].astype(object).where(X[c].isin(top), "other").astype("category")
print(X.shape, "| positives:", round(y.mean(), 4))

In [9]:
u = rng.random(len(X))
fit, tau_split, sn = u < 1 / 3, (u >= 1 / 3) & (u < 1 / 2), u >= 1 / 2

clf = HistGradientBoostingClassifier(
    categorical_features="from_dtype",  # requires scikit-learn >= 1.4
    max_iter=300,
    learning_rate=0.06,
    random_state=SEED,
).fit(X[fit], y[fit])

p = clf.predict_proba(X)[:, 1]
print("AUC (Sn) =", round(roc_auc_score(y[sn], p[sn]), 3))

AUC (Sn) = 0.631


In [10]:
# Threshold fitted on its own slice (Youden's J), then absorbed into the head's bias
fpr, tpr, thr = roc_curve(y[tau_split], p[tau_split])
tau = float(np.clip(thr[np.argmax(tpr - fpr)], 1e-6, 1 - 1e-6))

s = logit(np.clip(p, 1e-6, 1 - 1e-6)) - logit(tau)
p_shift = expit(s)
sgp_df = pd.DataFrame(
    {
        "y_true": y.astype(float),
        "y_pred": (s >= 0).astype(float),
        "kappa": np.maximum(p_shift, 1 - p_shift),  # softmax response
    }
)[sn].reset_index(drop=True)
print("tau =", round(tau, 4))

tau = 0.0978


In [11]:
def report(df, name):
    n = len(df)
    err = (df.y_pred != df.y_true).mean()
    fp = ((df.y_pred == 1) & (df.y_true == 0)).sum()
    fn = ((df.y_pred == 0) & (df.y_true == 1)).sum()
    ppv = ((df.y_pred == 1) & (df.y_true == 1)).sum() / max((df.y_pred == 1).sum(), 1)
    print(
        f"{name}: N={n} | positives={df.y_true.mean():.3f} | 0/1 risk={err:.3f} | "
        f"FP={fp} FN={fn} | FPR={fp/(df.y_true==0).sum():.3f} "
        f"FNR={fn/(df.y_true==1).sum():.3f} | PPV={ppv:.3f} | "
        f"kappa in [{df.kappa.min():.3f}, {df.kappa.max():.3f}]"
    )


report(sgp_df, "sgp_set")
pickle.dump(sgp_df, open(OUT, "wb"))
sgp_df.head(3)

sgp_set: N=34917 | positives=0.090 | 0/1 risk=0.322 | FP=9599 FN=1646 | FPR=0.302 FNR=0.525 | PPV=0.134 | kappa in [0.500, 0.931]


,y_true,y_pred,kappa
0,0.0,0.0,0.581275
1,0.0,0.0,0.568168
2,0.0,0.0,0.622939
